# `etl_main.ipynb` — Script principal de ejecución

Es el orquestador: no define
parámetros (eso es `config.ipynb`) ni funciones de limpieza (eso es
`transform.ipynb`) — los carga a ambos con `%run` y se limita a encadenar las
fases **Extract → Transform → Load** llamando a las funciones ya definidas.

**Migración Fashion Shop S.A.** — SQL Server (`BASE_DATOS_TIENDA_ROPAS_MARCAS`)
→ PostgreSQL (`db_fashionshop_dw`), aplicando RF-01 a RF-06.

| Fase | Contenido |
|---|---|
| 0 | Carga de `config.ipynb` y `transform.ipynb`, librerías propias, carpetas y logging |
| 1 | Extracción (SQL Server, con contingencia local para pruebas) |
| 2 | Transformación y limpieza (RF-01, RF-02, RF-03) — usa las funciones de `transform.ipynb` |
| 3 | Integridad referencial y separación válidos / cuarentena |
| 4 | Despliegue del esquema destino (DDL) y conexión a PostgreSQL |
| 5 | Carga idempotente (RF-06) de dimensiones y hechos |
| 6 | Cuarentena (RF-04) y auditoría de la ejecución (RF-05) |
| Extra | Demostración de triggers de auditoría e idempotencia |


---
## Fase 0 — Carga de módulos, librerías propias, carpetas y logging

### `%run` — el "import" de los notebooks

`config.ipynb` y `transform.ipynb` se ejecutan aquí con `%run`. Es una *magic*
de IPython (no Python puro, por eso empieza por `%`) que ejecuta el notebook
indicado y vuelca todas sus variables y funciones en **este** notebook, tal y
como haría `from modulo import *` con un `.py`. El orden importa:
`transform.ipynb` ya carga `config.ipynb` internamente, así que en principio
bastaría con un único `%run transform.ipynb` — pero se hacen explícitos los
dos para que quede claro, con solo mirar esta celda, de qué depende
`etl_main.ipynb`.

In [45]:
%run config.ipynb
%run transform.ipynb


Librerías de configuración importadas correctamente
Proyecto  : c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega
SQL Server: EM2026008667/BASE_DATOS_TIENDA_ROPAS_MARCAS
PostgreSQL: localhost:5432/db_fashionshop_dw (usuario: admin_etl)
--- config.ipynb cargado correctamente ---
Proyecto  : c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega
SQL Server: EM2026008667/BASE_DATOS_TIENDA_ROPAS_MARCAS
PostgreSQL: localhost:5432/db_fashionshop_dw (usuario: admin_etl)
Tasa USD->EUR: 1.15 | Categorías de moda: 5
Librerías de configuración importadas correctamente
Proyecto  : c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega
SQL Server: EM2026008667/BASE_DATOS_TIENDA_ROPAS_MARCAS
PostgreSQL: localhost:5432/db_fashionshop_dw (usuario: admin_etl)
--- config.ipynb cargado correctamente ---
Proyecto  : c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega
SQL Server: EM2026008667/BASE_DATOS_TIENDA_ROPAS_MARCAS
PostgreSQL: 

### Librerías propias de `etl_main`

`os`, `re`, `pandas`, `numpy` y `logging` ya están disponibles gracias al
`%run` anterior. Lo único que falta importar aquí es lo que **solo** usa la
orquestación E-T-L: el conector a SQLite (contingencia local), la
construcción de la cadena de conexión ODBC, las marcas de tiempo de la
ejecución, y el motor de conexión a las bases de datos.

In [46]:
import sqlite3
import urllib.parse
from datetime import datetime
from sqlalchemy import create_engine, text

print('Librerías de etl_main importadas correctamente')


Librerías de etl_main importadas correctamente


### Carpetas y logging

Crea la estructura de carpetas necesaria (si no existe) y arranca
`logs/ejecucion_etl.log`. Se aplica el formato (`fecha - NIVEL - mensaje`) con `filemode='w'`
(log limpio en cada ejecución) y limpieza previa de *handlers* para que, al
re-ejecutar celdas en Jupyter, no se dupliquen las líneas.

In [47]:
for carpeta in [DIR_PROYECTO, DIR_SQL, DIR_LOGS, DIR_RESULTADOS]:
    os.makedirs(carpeta, exist_ok=True)

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename=RUTA_LOG,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    filemode='w',   # log limpio en cada ejecución
    force=True,
)

hora_inicio_proceso = datetime.now()

logging.info('=' * 70)
logging.info('INICIO PROCESO ETL - Fashion Shop S.A. - Francisco Ortega')
logging.info(f'DIR_PROYECTO: {DIR_PROYECTO} | Origen: {SQL_DATABASE} | Destino: {PG_DATABASE}')
logging.info('=' * 70)

print(f'Log arrancado en: {RUTA_LOG}')


Log arrancado en: c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega\logs\ejecucion_etl.log


---
## Fase 1 — Extracción

### Conexión a SQL Server y lectura de las 6 tablas origen

Se conecta a `BASE_DATOS_TIENDA_ROPAS_MARCAS` con `pyodbc`/`SQLAlchemy`
(autenticación integrada de Windows, igual que en el resto de scripts del
equipo) y se leen las 6 tablas del origen.

**Contingencia para pruebas:** si SQL Server no está accesible desde esta
máquina (por ejemplo, en un entorno de validación sin ese servidor), el
`except` no detiene el proceso: registra el aviso en el log y cae a una
réplica local (`sample_data/origen_local_demo.db`, generada a partir del
propio script `tablas_origen_sql_server_proyecto_1_tienda_ropa.sql`) para
poder demostrar el pipeline completo igualmente. **En la máquina del empleado,
con acceso real al SQL Server, el `try` se
completa con éxito y esta contingencia nunca se activa.**


In [48]:
logging.info('[E] EXTRACCION INICIADA')

df_categorias = df_productos = df_clientes = df_vendedores = df_ubicaciones = df_ventas = None
modo_extraccion = None

try:
    logging.info('[E] Conectando a SQL Server origen')
    if SQL_USER:
        odbc_str = (
            f'DRIVER={SQL_DRIVER};SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};'
            f'UID={SQL_USER};PWD={SQL_PASSWORD};TrustServerCertificate=yes;'
        )
    else:
        odbc_str = (
            f'DRIVER={SQL_DRIVER};SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};'
            f'Trusted_Connection=yes;TrustServerCertificate=yes;'
        )
    params = urllib.parse.quote_plus(odbc_str)
    engine_origen = create_engine(f'mssql+pyodbc:///?odbc_connect={params}')

    with engine_origen.connect() as conn:
        conn.execute(text('SELECT 1'))
    logging.info(f'Conexion establecida: {SQL_SERVER}/{SQL_DATABASE}')
    print(f'Conexión a SQL Server ({SQL_DATABASE}) OK')

    df_categorias  = pd.read_sql('SELECT * FROM dbo.Categorias', engine_origen)
    df_productos   = pd.read_sql('SELECT * FROM dbo.Productos', engine_origen)
    df_clientes    = pd.read_sql('SELECT * FROM dbo.Clientes', engine_origen)
    df_vendedores  = pd.read_sql('SELECT * FROM dbo.Vendedores', engine_origen)
    df_ubicaciones = pd.read_sql('SELECT * FROM dbo.Ubicaciones', engine_origen)
    df_ventas      = pd.read_sql('SELECT * FROM dbo.Ventas', engine_origen)
    modo_extraccion = 'SQL_SERVER'

except Exception as e:
    logging.warning(f'No se pudo conectar a SQL Server ({e})')
    print(f'Aviso: SQL Server no disponible desde este entorno ({e})')

    if os.path.exists(RUTA_ORIGEN_LOCAL):
        modo_extraccion = 'LOCAL_CONTINGENCIA'
        logging.warning(f'MODO CONTINGENCIA: extrayendo desde réplica local {RUTA_ORIGEN_LOCAL}')
        print(f'Usando réplica local de origen para poder continuar: {RUTA_ORIGEN_LOCAL}')
        conn_local = sqlite3.connect(RUTA_ORIGEN_LOCAL)
        df_categorias  = pd.read_sql('SELECT * FROM Categorias', conn_local)
        df_productos   = pd.read_sql('SELECT * FROM Productos', conn_local)
        df_clientes    = pd.read_sql('SELECT * FROM Clientes', conn_local)
        df_vendedores  = pd.read_sql('SELECT * FROM Vendedores', conn_local)
        df_ubicaciones = pd.read_sql('SELECT * FROM Ubicaciones', conn_local)
        df_ventas      = pd.read_sql('SELECT * FROM Ventas', conn_local)
        conn_local.close()
    else:
        logging.critical('Sin SQL Server y sin réplica local disponible. Proceso detenido.')
        raise

registros_leidos = sum(len(d) for d in [df_categorias, df_productos, df_clientes,
                                         df_vendedores, df_ubicaciones, df_ventas])
logging.info(f'Extraccion completa ({modo_extraccion}): {registros_leidos} registros en 6 tablas')
print(f'\nModo de extracción: {modo_extraccion}')
print(f'Categorias: {len(df_categorias)} | Productos: {len(df_productos)} | Clientes: {len(df_clientes)}')
print(f'Vendedores: {len(df_vendedores)} | Ubicaciones: {len(df_ubicaciones)} | Ventas: {len(df_ventas)}')
print(f'Total registros leídos: {registros_leidos}')


Conexión a SQL Server (BASE_DATOS_TIENDA_ROPAS_MARCAS) OK

Modo de extracción: SQL_SERVER
Categorias: 8 | Productos: 7 | Clientes: 7
Vendedores: 4 | Ubicaciones: 4 | Ventas: 10000
Total registros leídos: 10030


---
## Fase 2 — Transformación y limpieza

Lista para acumular todas las filas rechazadas (`tabla_origen`, `datos_raw`,
`motivo_rechazo`) que, al final de esta fase, se cargarán en
`tb_errores_migracion` (RF-04). **Ningún error de una fila detiene el
proceso global**: se captura, se registra y se continúa con la siguiente.


In [49]:
errores = []  # lista de tuplas (tabla_origen, datos_raw, motivo_rechazo)


### RF-02: catálogo de moda y saneamiento de precios

1. Se descartan los productos de categorías que no son de moda (Hamburguesas,
   Pizzas, Sándwiches) — es un filtro de **alcance de negocio**, no un error de
   calidad, así que no se manda a cuarentena: se registra solo en el log.
2. A los productos de moda restantes se les parsea `Precio_Compra` y
   `Precio_Venta` con `parsear_importe` (símbolos, comas/puntos, USD→EUR).
   Un precio que no se puede convertir sí es un error de calidad → cuarentena.


*(la función `parsear_importe` está definida en `transform.ipynb`, ya cargado en la Fase 0)*

In [50]:
logging.info('[T] RF-02: filtrado de catalogo de moda')

df_categorias['es_moda'] = df_categorias['NombreCategoria'].isin(CATEGORIAS_MODA)
categorias_moda_ids = set(df_categorias.loc[df_categorias['es_moda'], 'CategoriaID'])

productos_moda = df_productos[df_productos['CategoriaID'].isin(categorias_moda_ids)].copy()
descartados_categoria = len(df_productos) - len(productos_moda)
logging.info(f'RF-02: {descartados_categoria} productos descartados por categoria fuera de alcance (no moda)')
print(f'Productos fuera de alcance (categoría no moda): {descartados_categoria}')
print(f'Productos de moda a procesar: {len(productos_moda)}')

logging.info('[T] RF-02: parseo financiero de precios (Precio_Compra / Precio_Venta)')

resultado_compra = productos_moda['Precio_Compra'].apply(lambda v: parsear_importe(v, 'Precio_Compra'))
resultado_venta  = productos_moda['Precio_Venta'].apply(lambda v: parsear_importe(v, 'Precio_Venta'))

productos_moda['precio_compra_eur'] = resultado_compra.apply(lambda x: x[0])
productos_moda['precio_venta_eur']  = resultado_venta.apply(lambda x: x[0])

for idx in productos_moda.index:
    _, motivo = resultado_compra.loc[idx]
    if motivo:
        errores.append(('Productos', f"ProductoID={productos_moda.loc[idx, 'ProductoID']} Precio_Compra='{productos_moda.loc[idx, 'Precio_Compra']}'", motivo))
    _, motivo = resultado_venta.loc[idx]
    if motivo:
        errores.append(('Productos', f"ProductoID={productos_moda.loc[idx, 'ProductoID']} Precio_Venta='{productos_moda.loc[idx, 'Precio_Venta']}'", motivo))

productos_ok = productos_moda.dropna(subset=['precio_compra_eur', 'precio_venta_eur']).copy()
productos_validos_ids = set(productos_ok['ProductoID'])

logging.info(f'RF-02: {len(productos_ok)} productos validos tras parseo financiero (de {len(productos_moda)} de moda)')
print(f'\nProductos válidos tras parseo financiero: {len(productos_ok)} / {len(productos_moda)}')
print(productos_ok[['ProductoID', 'NombreProducto', 'Precio_Compra', 'precio_compra_eur',
                     'Precio_Venta', 'precio_venta_eur']].to_string(index=False))


Productos fuera de alcance (categoría no moda): 2
Productos de moda a procesar: 5

Productos válidos tras parseo financiero: 5 / 5
 ProductoID                     NombreProducto Precio_Compra  precio_compra_eur Precio_Venta  precio_venta_eur
          1           Camisa de Algodón - Nike       25.00 €              25.00        35,00             35.00
          2   Pantalones de Mezclilla - Levi's     USD 40.00              34.78      65.50 €             65.50
          3         Chaqueta de Cuero - Adidas        150,00             150.00   220.00 USD            191.30
          4          Zapatos de Cuero - Clarks      100.00 €             100.00     145,99 €            145.99
          5 Cinturón de Cuero - Tommy Hilfiger         30.00              30.00        45.00             45.00


### RF-01: desacoplamiento y validación de clientes

`Cliente_Data` ("García Pérez, Juan - 12345678Z") se separa por regex en
`nombre`, `apellidos` y `dni`. Un DNI ausente, con formato inválido, o un
`Cliente_Data` totalmente irreconocible mandan la fila a cuarentena.


*(la función `parsear_cliente` está definida en `transform.ipynb`)*

In [51]:
logging.info('[T] RF-01: desacoplamiento y validacion de Cliente_Data')

resultado_cliente = df_clientes['Cliente_Data'].apply(parsear_cliente)
df_clientes['nombre']    = resultado_cliente.apply(lambda x: x[0])
df_clientes['apellidos'] = resultado_cliente.apply(lambda x: x[1])
df_clientes['dni']       = resultado_cliente.apply(lambda x: x[2])
df_clientes['_motivo']   = resultado_cliente.apply(lambda x: x[3])

for _, fila in df_clientes[df_clientes['_motivo'].notna()].iterrows():
    errores.append(('Clientes', f"ClienteID={fila['ClienteID']} Cliente_Data='{fila['Cliente_Data']}'", fila['_motivo']))

# Limpieza ligera del email (espacios); no se valida formato porque el enunciado
# (RF-01) solo exige validar el DNI - se documenta como asunción en la Memoria.
df_clientes['Email'] = df_clientes['Email'].apply(lambda e: str(e).strip() if pd.notna(e) else None)

clientes_ok = df_clientes[df_clientes['_motivo'].isna()].copy()
clientes_validos_ids = set(clientes_ok['ClienteID'])

logging.info(f'RF-01: {len(clientes_ok)} clientes validos | {df_clientes["_motivo"].notna().sum()} en cuarentena')
print(f'Clientes válidos: {len(clientes_ok)} / {len(df_clientes)}')
print(clientes_ok[['ClienteID', 'nombre', 'apellidos', 'dni', 'Email', 'Pais']].to_string(index=False))
print('\nClientes en cuarentena:')
print(df_clientes[df_clientes['_motivo'].notna()][['ClienteID', 'Cliente_Data', '_motivo']].to_string(index=False))


Clientes válidos: 4 / 7
 ClienteID nombre    apellidos       dni                   Email   Pais
         1   Juan García Pérez 12345678Z   juan.garcia@gmail.com España
         2  María    Rodríguez 87654321A   maria_rod@hotmail.com México
         5  Pedro     Martínez 45678912B       pedro.m@gmail.com  Chile
         6  Sofia        Gómez 98765432C sofia.gomez@outlook.com   Perú

Clientes en cuarentena:
 ClienteID                       Cliente_Data                                                                _motivo
         3 López D. Carlos - DNI_INVALIDO_999 Formato Cliente_Data no reconocido: López D. Carlos - DNI_INVALIDO_999
         4                     Fernández, Ana                                            DNI ausente: Fernández, Ana
         7                       UNKNOWN_USER                       Formato Cliente_Data no reconocido: UNKNOWN_USER


### RF-03: fechas e importes de ventas

`Fecha` se estandariza a ISO (`parsear_fecha`) y `MontoTotal_Raw` se limpia y
convierte a EUR con la misma función `parsear_importe` del Bloque 5 (reutilizada,
principio DRY). Fecha inválida, importe nulo o importe negativo → cuarentena.


*(reutiliza `parsear_fecha` y `parsear_importe` de `transform.ipynb`)*

In [52]:
logging.info('[T] RF-03: estandarizacion de fechas e importes de Ventas')

resultado_fecha = df_ventas['Fecha'].apply(parsear_fecha)
resultado_monto = df_ventas['MontoTotal_Raw'].apply(lambda v: parsear_importe(v, 'MontoTotal_Raw'))

df_ventas['fecha_parseada'] = resultado_fecha.apply(lambda x: x[0])
df_ventas['_motivo_fecha']  = resultado_fecha.apply(lambda x: x[1])
df_ventas['monto_eur']      = resultado_monto.apply(lambda x: x[0])
df_ventas['_motivo_monto']  = resultado_monto.apply(lambda x: x[1])

for _, fila in df_ventas[df_ventas['_motivo_fecha'].notna()].iterrows():
    errores.append(('Ventas', f"VentaID={fila['VentaID']} Fecha='{fila['Fecha']}'", fila['_motivo_fecha']))
for _, fila in df_ventas[df_ventas['_motivo_monto'].notna()].iterrows():
    errores.append(('Ventas', f"VentaID={fila['VentaID']} MontoTotal_Raw='{fila['MontoTotal_Raw']}'", fila['_motivo_monto']))

n_fecha_ko = df_ventas['_motivo_fecha'].notna().sum()
n_monto_ko = df_ventas['_motivo_monto'].notna().sum()
logging.info(f'RF-03: fechas invalidas={n_fecha_ko} | importes invalidos={n_monto_ko}')
print(f'Fechas inválidas: {n_fecha_ko} | Importes inválidos (nulos/negativos/no numéricos): {n_monto_ko}')
print('\n--- Muestra tras limpieza ---')
print(df_ventas[['VentaID', 'Fecha', 'fecha_parseada', 'MontoTotal_Raw', 'monto_eur']].head(8).to_string(index=False))


Fechas inválidas: 2500 | Importes inválidos (nulos/negativos/no numéricos): 497

--- Muestra tras limpieza ---
 VentaID        Fecha fecha_parseada MontoTotal_Raw  monto_eur
       1   09/08/2026     2026-08-09         21.0 €      21.00
       2   08/08/2026     2026-08-08         22.0 €      22.00
       3 INVALID_DATE           None       18.5 USD      16.09
       4   2026-08-06     2026-08-06         24.0 €      24.00
       5   05/08/2026     2026-08-05         25.0 €      25.00
       6   08/04/2026     2026-04-08       21.5 USD      18.70
       7 INVALID_DATE           None         27.0 €      27.00
       8   2026-08-02     2026-08-02         28.0 €      28.00


---
## Fase 3 — Integridad referencial y separación válidos / cuarentena

Una venta solo es válida si, además de tener fecha e importe correctos,
apunta a un **producto de moda válido** y a un **cliente válido** (los que
sobrevivieron a RF-01 y RF-02). Si no, también va a cuarentena: de lo
contrario `fact_ventas` tendría claves foráneas rotas.


In [53]:
logging.info('[T] Verificacion de integridad referencial de Ventas (producto/cliente)')

mask_producto_invalido = ~df_ventas['ProductoID'].isin(productos_validos_ids)
mask_cliente_invalido  = ~df_ventas['ClienteID'].isin(clientes_validos_ids)

for _, fila in df_ventas[mask_producto_invalido].iterrows():
    errores.append(('Ventas', f"VentaID={fila['VentaID']} ProductoID={fila['ProductoID']}",
                     'Producto fuera de alcance (categoría no moda) o inexistente'))
for _, fila in df_ventas[mask_cliente_invalido].iterrows():
    errores.append(('Ventas', f"VentaID={fila['VentaID']} ClienteID={fila['ClienteID']}",
                     'Cliente en cuarentena (RF-01) o inexistente'))

mask_venta_valida = (
    df_ventas['_motivo_fecha'].isna() & df_ventas['_motivo_monto'].isna()
    & ~mask_producto_invalido & ~mask_cliente_invalido
)
ventas_ok = df_ventas[mask_venta_valida].copy()
ventas_ko = df_ventas[~mask_venta_valida].copy()

# DataFrame consolidado de errores para la carga en tb_errores_migracion (RF-04)
df_errores = pd.DataFrame(errores, columns=['tabla_origen', 'datos_raw', 'motivo_rechazo'])
df_errores['fecha_deteccion'] = hora_inicio_proceso.date()

logging.info(f'Ventas validas: {len(ventas_ok)} | Ventas en cuarentena: {len(ventas_ko)}')
logging.info(f'Total filas de cuarentena (Productos+Clientes+Ventas): {len(df_errores)}')

print(f'Ventas válidas    : {len(ventas_ok)} / {len(df_ventas)}')
print(f'Ventas en cuarentena: {len(ventas_ko)}')
print(f'Total filas de cuarentena a evaluar (RF-04): {len(df_errores)}')

exportar_csv(ventas_ok, 'Ventas_Validas.csv')
exportar_csv(ventas_ko, 'Ventas_Invalidas.csv')
exportar_csv(df_errores, 'tb_errores_migracion.csv')


Ventas válidas    : 3035 / 10000
Ventas en cuarentena: 6965
Total filas de cuarentena a evaluar (RF-04): 10142
   Guardado: Ventas_Validas.csv (3035 filas)
   Guardado: Ventas_Invalidas.csv (6965 filas)
   Guardado: tb_errores_migracion.csv (10142 filas)


---
## Fase 4 — Conexión a PostgreSQL y despliegue del esquema destino

Se conecta al PostgreSQL levantado por `docker-compose.yml`
(`db_fashionshop_dw`) y se aplica `sql/ddl_destino_postgresql.sql`. El DDL usa
`CREATE TABLE IF NOT EXISTS` / `CREATE OR REPLACE`, así que aplicarlo varias
veces es seguro y no destruye datos ya cargados.


In [54]:
logging.info('[L] FASE 4 - Conexion a PostgreSQL')

engine_destino = None
try:
    engine_destino = create_engine(
        f'postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}'
    )
    with engine_destino.connect() as conn:
        conn.execute(text('SELECT 1'))
    logging.info(f'Conexion establecida: {PG_HOST}:{PG_PORT}/{PG_DATABASE}')
    print(f'Conexión a PostgreSQL ({PG_DATABASE}) OK')

    with open(RUTA_DDL, encoding='utf-8') as f:
        ddl_sql = f.read()
    with engine_destino.begin() as conn:
        # exec_driver_sql envía el script completo tal cual al driver (psycopg2),
        # que sí soporta varias sentencias separadas por ';' en una sola llamada.
        # Trocear el script nosotros mismos por ';' rompería las funciones
        # PL/pgSQL (su cuerpo, entre $$...$$, contiene punto y coma internos).
        conn.exec_driver_sql(ddl_sql)
    logging.info('DDL aplicado correctamente sobre db_fashionshop_dw')
    print('Esquema destino (tablas, índices, triggers) verificado/creado a partir de ddl_destino_postgresql.sql')

except Exception as e:
    logging.error(f'Error de conexion o de aplicacion del DDL en PostgreSQL: {e}')
    print(f'Error: {e}')
    print('No se podrá completar la carga. Revisa que el contenedor de docker-compose esté levantado.')


Conexión a PostgreSQL (db_fashionshop_dw) OK
Esquema destino (tablas, índices, triggers) verificado/creado a partir de ddl_destino_postgresql.sql


---
## Fase 5 — Carga de dimensiones y hechos (idempotente, RF-06)

Cada tabla se carga con `upsert_dataframe` (Bloque 3): `INSERT ... ON CONFLICT
DO UPDATE` sobre la clave de origen. Reejecutar esta celda las veces que sea
no duplica filas — actualiza las existentes con los mismos valores.


*(usa `upsert_dataframe`, definida en `transform.ipynb`)*

In [55]:
logging.info('[L] FASE 5 - Carga de dimensiones y hechos')
registros_insertados = 0

print('Dimensiones sin transformación (carga directa):')
registros_insertados += upsert_dataframe(
    df_categorias[['CategoriaID', 'NombreCategoria']].rename(
        columns={'CategoriaID': 'categoria_id', 'NombreCategoria': 'nombre_categoria'}),
    'dim_categorias', ['categoria_id'], engine_destino)

registros_insertados += upsert_dataframe(
    df_vendedores.rename(columns={'VendedorID': 'vendedor_id', 'NombreVendedor': 'nombre_vendedor', 'Pais': 'pais'}),
    'dim_vendedores', ['vendedor_id'], engine_destino)

registros_insertados += upsert_dataframe(
    df_ubicaciones.rename(columns={'UbicacionID': 'ubicacion_id', 'Ciudad': 'ciudad', 'Pais': 'pais'}),
    'dim_ubicaciones', ['ubicacion_id'], engine_destino)

print('\nDimensiones transformadas (RF-01 / RF-02):')
registros_insertados += upsert_dataframe(
    productos_ok[['ProductoID', 'NombreProducto', 'CategoriaID', 'precio_compra_eur', 'precio_venta_eur', 'Stock']]
        .rename(columns={'ProductoID': 'producto_id', 'NombreProducto': 'nombre_producto',
                          'CategoriaID': 'categoria_id', 'Stock': 'stock'}),
    'dim_productos', ['producto_id'], engine_destino)

registros_insertados += upsert_dataframe(
    clientes_ok[['ClienteID', 'nombre', 'apellidos', 'dni', 'Email', 'Pais']]
        .rename(columns={'ClienteID': 'cliente_id', 'Email': 'email', 'Pais': 'pais'}),
    'dim_clientes', ['cliente_id'], engine_destino)

print('\nTabla de hechos:')
registros_insertados += upsert_dataframe(
    ventas_ok[['VentaID', 'fecha_parseada', 'ProductoID', 'VendedorID', 'ClienteID', 'UbicacionID', 'Cantidad', 'monto_eur']]
        .rename(columns={'VentaID': 'venta_id_origen', 'fecha_parseada': 'fecha', 'ProductoID': 'producto_id',
                          'VendedorID': 'vendedor_id', 'ClienteID': 'cliente_id', 'UbicacionID': 'ubicacion_id',
                          'Cantidad': 'cantidad', 'monto_eur': 'monto_total_eur'}),
    'fact_ventas', ['venta_id_origen'], engine_destino)

logging.info(f'FASE 5 completada: {registros_insertados} registros upsert en dimensiones y hechos')
print(f'\nTotal registros insertados/actualizados: {registros_insertados}')


Dimensiones sin transformación (carga directa):
   PostgreSQL: dim_categorias -> 8 registros (upsert)
   PostgreSQL: dim_vendedores -> 4 registros (upsert)
   PostgreSQL: dim_ubicaciones -> 4 registros (upsert)

Dimensiones transformadas (RF-01 / RF-02):
   PostgreSQL: dim_productos -> 5 registros (upsert)
   PostgreSQL: dim_clientes -> 4 registros (upsert)

Tabla de hechos:
   PostgreSQL: fact_ventas -> 3035 registros (upsert)

Total registros insertados/actualizados: 3060


---
## Fase 6 — Cuarentena (RF-04) y auditoría de la ejecución (RF-05)

Se cargan los errores acumulados en `tb_errores_migracion` y, al cerrar el
proceso, se inserta una fila en `tb_ejecuciones_etl` con el resumen de la
corrida (inicio, fin, leídos, insertados, errores) — la auditoría de
*ejecuciones*, distinta de la auditoría de *operaciones* (Bloque de triggers).


*(usa `insertar_cuarentena`, definida en `transform.ipynb`)*

In [56]:
logging.info('[L] FASE 6 - Carga de cuarentena')
registros_error = insertar_cuarentena(df_errores, engine_destino)

hora_fin_proceso = datetime.now()
duracion = (hora_fin_proceso - hora_inicio_proceso).total_seconds()

try:
    with engine_destino.begin() as conn:
        conn.execute(text("""
            INSERT INTO tb_ejecuciones_etl (fecha_inicio, fecha_fin, registros_leidos, registros_insertados, registros_error)
            VALUES (:inicio, :fin, :leidos, :insertados, :errores)
        """), {
            'inicio': hora_inicio_proceso, 'fin': hora_fin_proceso,
            'leidos': registros_leidos, 'insertados': registros_insertados, 'errores': registros_error,
        })
    logging.info(f'tb_ejecuciones_etl actualizada. Duracion total: {duracion:.2f}s')
    print(f'Ejecución registrada en tb_ejecuciones_etl (duración: {duracion:.2f} s)')
except Exception as e:
    logging.error(f'Error registrando tb_ejecuciones_etl: {e}')
    print(f'Error: {e}')


   Cuarentena: 10142 filas evaluadas (duplicados ya existentes se ignoran)
Ejecución registrada en tb_ejecuciones_etl (duración: 2.07 s)


---
## Demostración de los triggers de auditoría (RF-05)

El DDL crea un trigger `AFTER INSERT OR UPDATE OR DELETE` por cada tabla de
negocio auditada (`dim_clientes`, `dim_productos`, `fact_ventas`), cada uno
apuntando a su tabla `auditoria_*`. Para demostrar que funcionan, se hace un
INSERT + UPDATE + DELETE de prueba sobre un cliente ficticio (`cliente_id =
999`, fuera del rango real 1-N de Fashion Shop) y se consulta el rastro que
queda en `auditoria_clientes`.


In [57]:
try:
    with engine_destino.begin() as conn:
        conn.execute(text("""
            INSERT INTO dim_clientes (cliente_id, nombre, apellidos, dni, email, pais)
            VALUES (999, 'Test', 'Auditoria', '11111111H', 'test@test.com', 'España')
        """))
        conn.execute(text("UPDATE dim_clientes SET pais = 'Portugal' WHERE cliente_id = 999"))
        conn.execute(text("DELETE FROM dim_clientes WHERE cliente_id = 999"))

    df_auditoria_demo = pd.read_sql(text("""
        SELECT audit_id, operacion, cliente_id,
               datos_anteriores->>'pais' AS pais_antes,
               datos_nuevos->>'pais'     AS pais_despues,
               fecha_operacion
        FROM auditoria_clientes
        WHERE cliente_id = 999
        ORDER BY audit_id
    """), engine_destino)

    logging.info(f'RF-05: demostracion de triggers OK, {len(df_auditoria_demo)} filas de auditoria generadas')
    print('Rastro de auditoría generado automáticamente por los triggers:')
    print(df_auditoria_demo.to_string(index=False))

except Exception as e:
    logging.error(f'Error en demostracion de triggers: {e}')
    print(f'Error: {e}')


Rastro de auditoría generado automáticamente por los triggers:
 audit_id operacion  cliente_id pais_antes pais_despues            fecha_operacion
        5    INSERT         999        NaN       España 2026-08-10 10:08:32.909667
        6    UPDATE         999     España     Portugal 2026-08-10 10:08:32.909667
        7    DELETE         999   Portugal          NaN 2026-08-10 10:08:32.909667
       12    INSERT         999        NaN       España 2026-08-11 08:10:26.648862
       13    UPDATE         999     España     Portugal 2026-08-11 08:10:26.648862
       14    DELETE         999   Portugal          NaN 2026-08-11 08:10:26.648862
       19    INSERT         999        NaN       España 2026-08-11 09:34:54.039705
       20    UPDATE         999     España     Portugal 2026-08-11 09:34:54.039705
       21    DELETE         999   Portugal          NaN 2026-08-11 09:34:54.039705
       26    INSERT         999        NaN       España 2026-08-11 09:35:43.281520
       27    UPDATE     

---
## Demostración de idempotencia (RF-06)

Se vuelve a ejecutar la carga de `fact_ventas` (misma llamada que en la Fase
5) y se compara el recuento de filas antes y después: si el proceso es
idempotente, el número **no debe cambiar**.


*(reutiliza `upsert_dataframe` una segunda vez, definida en `transform.ipynb`)*

In [58]:
with engine_destino.connect() as conn:
    conteo_antes = conn.execute(text('SELECT COUNT(*) FROM fact_ventas')).scalar()

_ = upsert_dataframe(
    ventas_ok[['VentaID', 'fecha_parseada', 'ProductoID', 'VendedorID', 'ClienteID', 'UbicacionID', 'Cantidad', 'monto_eur']]
        .rename(columns={'VentaID': 'venta_id_origen', 'fecha_parseada': 'fecha', 'ProductoID': 'producto_id',
                          'VendedorID': 'vendedor_id', 'ClienteID': 'cliente_id', 'UbicacionID': 'ubicacion_id',
                          'Cantidad': 'cantidad', 'monto_eur': 'monto_total_eur'}),
    'fact_ventas', ['venta_id_origen'], engine_destino)

with engine_destino.connect() as conn:
    conteo_despues = conn.execute(text('SELECT COUNT(*) FROM fact_ventas')).scalar()

logging.info(f'RF-06: idempotencia verificada. fact_ventas antes={conteo_antes} despues={conteo_despues}')
print(f'fact_ventas antes de reejecutar : {conteo_antes} filas')
print(f'fact_ventas despues de reejecutar: {conteo_despues} filas')
print('IDEMPOTENTE: no se generaron duplicados' if conteo_antes == conteo_despues else 'ATENCION: el recuento cambió')


   PostgreSQL: fact_ventas -> 3035 registros (upsert)
fact_ventas antes de reejecutar : 3035 filas
fact_ventas despues de reejecutar: 3035 filas
IDEMPOTENTE: no se generaron duplicados


---
## Resumen final

In [59]:
print('\n' + '=' * 70)
print('RESUMEN ETL - Fashion Shop S.A. - Francisco Ortega')
print('=' * 70)

print(f'\n[E] Extracción ({modo_extraccion}): {registros_leidos} registros en 6 tablas origen')

print(f'\n[T] RF-02 Productos de moda válidos : {len(productos_ok)} / {len(df_productos)}'
      f' ({descartados_categoria} fuera de alcance)')
print(f'    RF-01 Clientes válidos            : {len(clientes_ok)} / {len(df_clientes)}')
print(f'    RF-03 Ventas válidas               : {len(ventas_ok)} / {len(df_ventas)}')

print(f'\n[L] Registros insertados/actualizados en PostgreSQL: {registros_insertados}')
print(f'    Filas evaluadas en tb_errores_migracion (RF-04)  : {registros_error}')
print(f'    Ejecución registrada en tb_ejecuciones_etl (RF-05)')
print(f'    Duración total: {duracion:.2f} s')

print(f'\nCSVs de respaldo en: {DIR_RESULTADOS}')
print(f'Log de ejecución en : {RUTA_LOG}')

logging.info('=' * 70)
logging.info('ETL FINALIZADO')
logging.info(f'Leidos: {registros_leidos} | Insertados: {registros_insertados} | Errores: {registros_error} | Duracion: {duracion:.2f}s')
logging.info('=' * 70)

print('\nETL completado con éxito.')



RESUMEN ETL - Fashion Shop S.A. - Francisco Ortega

[E] Extracción (SQL_SERVER): 10030 registros en 6 tablas origen

[T] RF-02 Productos de moda válidos : 5 / 7 (2 fuera de alcance)
    RF-01 Clientes válidos            : 4 / 7
    RF-03 Ventas válidas               : 3035 / 10000

[L] Registros insertados/actualizados en PostgreSQL: 3060
    Filas evaluadas en tb_errores_migracion (RF-04)  : 10142
    Ejecución registrada en tb_ejecuciones_etl (RF-05)
    Duración total: 2.07 s

CSVs de respaldo en: c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega\Resultados
Log de ejecución en : c:\Users\EM2026008667\Entregable_Proyecto_Tienda_Ropa_Francisco_Ortega\logs\ejecucion_etl.log

ETL completado con éxito.
